# Chinook sample DB, using SQLAlchemy module

---


In [69]:
# ============================================
# Libraries and Inicial Set UP
# ============================================

# Core Python
import os
import urllib.request

# Data handling
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# SQLAlchemy imports
import sqlalchemy
from sqlalchemy import create_engine, inspect, func, MetaData
from sqlalchemy.ext.automap import automap_base
from sqlalchemy.orm import sessionmaker

# Display settings - use a style that works across matplotlib versions
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('seaborn-whitegrid')  # Fallback for older matplotlib

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_context("paper", font_scale=0.8)


print("All libraries loaded!")
print(f"SQLAlchemy version: {sqlalchemy.__version__}")

All libraries loaded!
SQLAlchemy version: 2.0.46


🌟 Exercise 1 : Open the database
* open the database using sqlalchemy module interface. Create an engine object in a variable named engine.
* call the connect() method to obtain a connection and place in a variable named cur.


In [70]:
# ============================
# Download Chinook sample database and connect DB
# ============================

chinook_url = "https://github.com/lerocha/chinook-database/raw/master/ChinookDatabase/DataSources/Chinook_Sqlite.sqlite"

if not os.path.exists("chinook.db"):
    print("Downloading chinook.db...")
    urllib.request.urlretrieve(chinook_url, "chinook.db")

assert os.path.exists("chinook.db")
print("Database ready.")



engine = create_engine('sqlite:///chinook.db')   #  Create the Engine
cur = engine.connect()
print(f"Connected to: {engine.url}")


Database ready.
Connected to: sqlite:///chinook.db


* now run the code below to to run reflection on the database, prepare classes that map to the database and create an orm session :

In [71]:
# ============================
# Reflection and ORM setup
# ============================

# useful: extract classes from the chinook database
metadata = sqlalchemy.MetaData()
metadata.reflect(engine)

## we need to do this once
from sqlalchemy.ext.automap import automap_base

# produce a set of mappings from this MetaData.
Base = automap_base(metadata=metadata)

# calling prepare() just sets up mapped classes and relationships.
Base.prepare()


# Create class aliases
# Note: Chinook uses PascalCase table names (Artist, Album, Track, etc.)
Artist = Base.classes.Artist
Album = Base.classes.Album
Track = Base.classes.Track
Genre = Base.classes.Genre
Customer = Base.classes.Customer
Invoice = Base.classes.Invoice
InvoiceItem = Base.classes.InvoiceLine  # Note: Table is called "InvoiceLine" in Chinook
Employee = Base.classes.Employee  # For practice exercises

# Create session ORM
Session = sessionmaker(bind=engine)
session = Session()
print("ORM Session ready!")


ORM Session ready!


In [72]:
print("Auto-generated classes:")
for class_name in sorted(Base.classes.keys()):
    print(f"  - {class_name}")


Auto-generated classes:
  - Album
  - Artist
  - Customer
  - Employee
  - Genre
  - Invoice
  - InvoiceLine
  - MediaType
  - Playlist
  - Track


In [73]:
# ============================
# Helper functions
# ============================

from IPython.display import display

def get_results(query):
    q = query.statement if hasattr(query, 'statement') else query
    return pd.read_sql(q, engine)

def display_results(query):
    df = get_results(query)
    display(df)
    return df

print("Helper functions ready!")

Helper functions ready!


🌟 Exercise 2 : table names
* print out all the table names

In [74]:
inspector = inspect(engine)
table_names = inspector.get_table_names()

print("Tables in Chinook DB:")
for i, table in enumerate(table_names):
  print(f" {i}. {table}")

# Otra opción
print("\nFrom metadata:", metadata.tables.keys())

Tables in Chinook DB:
 0. Album
 1. Artist
 2. Customer
 3. Employee
 4. Genre
 5. Invoice
 6. InvoiceLine
 7. MediaType
 8. Playlist
 9. PlaylistTrack
 10. Track

From metadata: dict_keys(['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'Track', 'MediaType', 'Playlist', 'PlaylistTrack'])


🌟 Exercise 3 : Tracks
* print out the first three tracks in the tracks table

In [75]:
query = session.query(Track).limit(3)

df_tracks = display_results(query)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99


🌟 Exercise 4 : Albums from Tracks
* print out the track name and albums title of the first 20 tracks in the tracks table

In [76]:
query = (
    session.query(
        Track.Name.label('track_name'),
        Album.Title.label('album_title')
    )
    .join(Album, Track.AlbumId == Album.AlbumId)
    .limit(20)
)
df_tracks_albums = display_results(query)

,track_name,album_title
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You
1,Balls to the Wall,Balls to the Wall
2,Fast As a Shark,Restless and Wild
3,Restless and Wild,Restless and Wild
4,Princess of the Dawn,Restless and Wild
5,Put The Finger On You,For Those About To Rock We Salute You
6,Let's Get It Up,For Those About To Rock We Salute You
7,Inject The Venom,For Those About To Rock We Salute You
8,Snowballed,For Those About To Rock We Salute You
9,Evil Walks,For Those About To Rock We Salute You


🌟 Exercise 5: Tracks sold
* print out the first 10 track sales from the invoice_items table
* for these first 10 sales, print what are the names of the track sold, and the quantity sold

In [77]:
# First 10 invoice items
query = session.query(
    InvoiceItem.InvoiceLineId,
    InvoiceItem.TrackId,
    InvoiceItem.Quantity,
    InvoiceItem.UnitPrice
).limit(10)

df_invoice_items = display_results(query)


,InvoiceLineId,TrackId,Quantity,UnitPrice
0,1,2,1,0.99
1,2,4,1,0.99
2,3,6,1,0.99
3,4,8,1,0.99
4,5,10,1,0.99
5,6,12,1,0.99
6,7,16,1,0.99
7,8,20,1,0.99
8,9,24,1,0.99
9,10,28,1,0.99


In [78]:
#Join with tracks to get names

query = (
    session.query(
        Track.Name.label('track_name'),
        InvoiceItem.Quantity.label('quantity'),
        InvoiceItem.UnitPrice.label('price')
    )
    .join(Track, InvoiceItem.TrackId == Track.TrackId)
    .limit(10)
)

df_tracks_sold = display_results(query)

,track_name,quantity,price
0,Balls to the Wall,1,0.99
1,Restless and Wild,1,0.99
2,Put The Finger On You,1,0.99
3,Inject The Venom,1,0.99
4,Evil Walks,1,0.99
5,Breaking The Rules,1,0.99
6,Dog Eat Dog,1,0.99
7,Overdose,1,0.99
8,Love In An Elevator,1,0.99
9,Janie's Got A Gun,1,0.99


🌟 Exercise 6 : Top tracks sold
* print the names of top 10 tracks sold, and how many they times they were sold

In [79]:
query = (
    session.query(
        Track.Name.label('track_name'),
        func.sum(InvoiceItem.Quantity).label('total_quantity_sold')
    )
    .join(InvoiceItem, InvoiceItem.TrackId == Track.TrackId)
    .group_by(Track.TrackId)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
)

df_top_tracks = display_results(query)

,track_name,total_quantity_sold
0,Balls to the Wall,2
1,Inject The Venom,2
2,Snowballed,2
3,Overdose,2
4,Deuces Are Wild,2
5,Not The Doctor,2
6,Por Causa De Você,2
7,Welcome Home (Sanitarium),2
8,Snowblind,2
9,Cornucopia,2


🌟 Exercise 7 : Top selling artists
* Who are the top 10 highest selling artists?

In [80]:
query = (
    session.query(
        Artist.Name.label('artist_name'),
        func.sum(InvoiceItem.Quantity).label('units_sold'),
        func.sum(InvoiceItem.UnitPrice * InvoiceItem.Quantity).label('revenue')
    )
    .join(Album, Album.ArtistId == Artist.ArtistId)
    .join(Track, Track.AlbumId == Album.AlbumId)
    .join(InvoiceItem, InvoiceItem.TrackId == Track.TrackId)
    .group_by(Artist.ArtistId)
    .order_by(func.sum(InvoiceItem.Quantity).desc())
    .limit(10)
)

df_top_artists = display_results(query)

,artist_name,units_sold,revenue
0,Iron Maiden,140,138.60
1,U2,107,105.93
2,Metallica,91,90.09
3,Led Zeppelin,87,86.13
4,Os Paralamas Do Sucesso,45,44.55
5,Deep Purple,44,43.56
6,Faith No More,42,41.58
7,Lost,41,81.59
8,Eric Clapton,40,39.60
9,R.E.M.,39,38.61
